In [ ]:
# =============================================================================
# 1. DATA PREPROCESSING AND FEATURE ENGINEERING FOR ML MODELS
# =============================================================================

print("="*80)
print("MACHINE LEARNING MODELS IMPLEMENTATION")
print("="*80)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

# Advanced ML Libraries
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# Time series libraries
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

print("Libraries imported successfully!")
print(f"LightGBM version: {lgb.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"CatBoost version: {cb.__version__}")

# Load the processed data
print("\nLoading processed data...")
refactored_df = pd.read_csv('../data/refactored_df.csv')
weather_data = pd.read_csv('../data/weather_data.csv')
trends_df = pd.read_csv('../data/customer_trends.csv')

# Convert date columns
refactored_df['Date'] = pd.to_datetime(refactored_df['Date'])
weather_data['Date'] = pd.to_datetime(weather_data['Date'])
trends_df['Date'] = pd.to_datetime(trends_df['Month'])

print(f"Data loaded successfully!")
print(f"Sales data shape: {refactored_df.shape}")
print(f"Weather data shape: {weather_data.shape}")
print(f"Trends data shape: {trends_df.shape}")


In [ ]:
main_df = (
    refactored_df.groupby([refactored_df['Date'].dt.to_period('M').dt.to_timestamp().rename('MonthStart'), 'Branch'])
    .agg({'Qty': 'sum'})
    .reset_index()
)
main_df.columns = ['Date', 'Branch', 'Qty']
main_df = main_df.merge(weather_data, on=['Date', 'Branch']).reset_index(drop=True)
main_df = main_df.merge(trends_df, on=['Date'])
main_df = main_df.drop(columns=['Month'])
main_df['Seasonality_Level'] = main_df['Date'].dt.month_name().str[:3].map({
    'Mar': 3, 'Dec': 3, 'Feb': 3, 'Apr': 3,
    'May': 2, 'Jan': 2,
    'Jun': 1, 'Jul': 1, 'Aug': 1, 'Sep': 1, 'Oct': 1, 'Nov': 1
})

In [ ]:
# =============================================================================
# 2. COMPREHENSIVE FEATURE ENGINEERING
# =============================================================================

print("\n2. COMPREHENSIVE FEATURE ENGINEERING")
print("-" * 50)

# Import required modules
from sklearn.preprocessing import LabelEncoder

def drop_nan_columns(df, threshold=1.0, verbose=True):
    """
    Drop columns with a fraction of NaN values above the given threshold.
    
    Parameters:
        df (pd.DataFrame): Input DataFrame.
        threshold (float): Fraction (0–1). Columns with NaN ratio >= threshold are dropped.
                           Default 1.0 means drop columns that are entirely NaN.
        verbose (bool): Whether to print information about dropped columns.
        
    Returns:
        pd.DataFrame: DataFrame with selected columns dropped.
        list: List of dropped column names.
    """
    # Calculate fraction of missing values per column
    nan_ratio = df.isna().mean()
    
    # Select columns to drop
    drop_cols = nan_ratio[nan_ratio >= threshold].index.tolist()
    
    # Drop them
    df_cleaned = df.drop(columns=drop_cols)
    
    if verbose:
        print(f"Dropped {len(drop_cols)} column(s) with ≥{threshold*100:.0f}% NaN values:")
        if drop_cols:
            print(drop_cols)
        else:
            print("No columns dropped.")
    
    return df_cleaned

def create_ml_features(df):
    """
    Create comprehensive features for machine learning models with proper NaN handling
    """
    print("Creating comprehensive features...")
    
    # Start with the base dataframe
    ml_df = df.copy()
    ml_df = drop_nan_columns(ml_df, 0.5)
    
    # Check data availability for each branch
    branch_counts = ml_df.groupby('Branch').size()
    print(f"  Data points per branch: {dict(branch_counts)}")
    
    # 1. TIME-BASED FEATURES
    print("  Creating time-based features...")
    ml_df['year'] = ml_df['Date'].dt.year
    ml_df['month'] = ml_df['Date'].dt.month
    ml_df['day'] = ml_df['Date'].dt.day
    ml_df['dayofweek'] = ml_df['Date'].dt.dayofweek
    ml_df['dayofyear'] = ml_df['Date'].dt.dayofyear
    ml_df['week'] = ml_df['Date'].dt.isocalendar().week
    ml_df['quarter'] = ml_df['Date'].dt.quarter
    
    # Cyclical encoding for time features
    ml_df['month_sin'] = np.sin(2 * np.pi * ml_df['month'] / 12)
    ml_df['month_cos'] = np.cos(2 * np.pi * ml_df['month'] / 12)
    ml_df['dayofweek_sin'] = np.sin(2 * np.pi * ml_df['dayofweek'] / 7)
    ml_df['dayofweek_cos'] = np.cos(2 * np.pi * ml_df['dayofweek'] / 7)
    ml_df['quarter_sin'] = np.sin(2 * np.pi * ml_df['quarter'] / 4)
    ml_df['quarter_cos'] = np.cos(2 * np.pi * ml_df['quarter'] / 4)
    
    # 2. LAG FEATURES (ADAPTIVE BASED ON DATA AVAILABILITY)
    print("  Creating lag features...")
    # Sort by date and branch for proper lag calculation
    ml_df = ml_df.sort_values(['Branch', 'Date'])
    
    # Use smaller lag periods appropriate for monthly data
    lag_periods = [1, 2, 3, 6, 12]  # months instead of days
    for lag in lag_periods:
        ml_df[f'qty_lag_{lag}m'] = ml_df.groupby('Branch')['Qty'].shift(lag)
        # Only create rolling mean if we have enough data
        if lag <= 3:  # Only for short lags
            ml_df[f'qty_lag_{lag}m_mean'] = ml_df.groupby('Branch')['Qty'].shift(lag).rolling(window=min(3, lag), min_periods=1).mean()
    
    # 3. ROLLING STATISTICS (ADAPTIVE WINDOWS)
    print("  Creating rolling statistics...")
    # Use smaller windows appropriate for monthly data
    rolling_windows = [2, 3, 6, 12]  # months
    for window in rolling_windows:
        # Use min_periods to handle insufficient data gracefully
        ml_df[f'qty_rolling_mean_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).mean().reset_index(0, drop=True)
        ml_df[f'qty_rolling_std_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).std().reset_index(0, drop=True)
        ml_df[f'qty_rolling_max_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).max().reset_index(0, drop=True)
        ml_df[f'qty_rolling_min_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).min().reset_index(0, drop=True)
        ml_df[f'qty_rolling_sum_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).sum().reset_index(0, drop=True)
    
    # 4. EXPANDING STATISTICS
    print("  Creating expanding statistics...")
    ml_df['qty_expanding_mean'] = ml_df.groupby('Branch')['Qty'].expanding().mean().reset_index(0, drop=True)
    ml_df['qty_expanding_std'] = ml_df.groupby('Branch')['Qty'].expanding().std().reset_index(0, drop=True)
    ml_df['qty_expanding_max'] = ml_df.groupby('Branch')['Qty'].expanding().max().reset_index(0, drop=True)
    ml_df['qty_expanding_min'] = ml_df.groupby('Branch')['Qty'].expanding().min().reset_index(0, drop=True)
    
    # 5. SEASONAL FEATURES
    print("  Creating seasonal features...")
    # Monthly seasonality
    monthly_avg = ml_df.groupby('month')['Qty'].mean()
    ml_df['monthly_seasonality'] = ml_df['month'].map(monthly_avg)
    
    # Quarterly seasonality
    quarterly_avg = ml_df.groupby('quarter')['Qty'].mean()
    ml_df['quarterly_seasonality'] = ml_df['quarter'].map(quarterly_avg)
    
    # Day of week seasonality
    dow_avg = ml_df.groupby('dayofweek')['Qty'].mean()
    ml_df['dow_seasonality'] = ml_df['dayofweek'].map(dow_avg)
    
    # 6. WEATHER LAG FEATURES (ADAPTIVE)
    print("  Creating weather lag features...")
    weather_lags = [1, 2, 3, 6]  # months
    for lag in weather_lags:
        ml_df[f'temp_lag_{lag}m'] = ml_df.groupby('Branch')['Avg Temp'].shift(lag)
        ml_df[f'humidity_lag_{lag}m'] = ml_df.groupby('Branch')['Avg Humidity'].shift(lag)
        ml_df[f'wind_lag_{lag}m'] = ml_df.groupby('Branch')['Avg Wind Speed'].shift(lag)
    
    # 7. TRENDS LAG FEATURES
    print("  Creating trends lag features...")
    trends_lags = [1, 2, 3, 6]
    for lag in trends_lags:
        ml_df[f'trends_lag_{lag}m'] = ml_df['Interest'].shift(lag)
    
    # 8. PRODUCT FEATURES
    print("  Creating product features...")
    # Branch encoding
    branch_encoder = LabelEncoder()
    ml_df['branch_encoded'] = branch_encoder.fit_transform(ml_df['Branch'])
    
    # 9. INTERACTION FEATURES
    print("  Creating interaction features...")
    ml_df['temp_humidity_interaction'] = ml_df['Avg Temp'] * ml_df['Avg Humidity']
    ml_df['temp_wind_interaction'] = ml_df['Avg Temp'] * ml_df['Avg Wind Speed']
    
    # 10. STATISTICAL FEATURES
    print("  Creating statistical features...")
    # Temperature statistics
    ml_df['temp_range'] = ml_df['Max Temp'] - ml_df['Min Temp']
    ml_df['humidity_range'] = ml_df['Max Humidity'] - ml_df['Min Humidity']
    ml_df['wind_range'] = ml_df['Max Wind Speed'] - ml_df['Min Wind Speed']
    
    # Temperature deviation from historical average
    temp_avg = ml_df.groupby('month')['Avg Temp'].transform('mean')
    ml_df['temp_deviation'] = ml_df['Avg Temp'] - temp_avg
    
    # Humidity deviation from historical average
    humidity_avg = ml_df.groupby('month')['Avg Humidity'].transform('mean')
    ml_df['humidity_deviation'] = ml_df['Avg Humidity'] - humidity_avg
    
    # 11. BUSINESS FEATURES
    print("  Creating business features...")
    # Days since last sale (convert to months for monthly data)
    ml_df['months_since_last_sale'] = ml_df.groupby('Branch')['Date'].diff().dt.days / 30.44  # Approximate days per month
    
    # Sales momentum (recent vs historical average) - handle division by zero
    ml_df['sales_momentum_3m'] = np.where(
        ml_df['qty_expanding_mean'] != 0,
        ml_df['qty_rolling_mean_3m'] / ml_df['qty_expanding_mean'],
        1.0  # Default to 1 if no historical data
    )
    ml_df['sales_momentum_6m'] = np.where(
        ml_df['qty_expanding_mean'] != 0,
        ml_df['qty_rolling_mean_6m'] / ml_df['qty_expanding_mean'],
        1.0
    )
    
    # Market share by branch
    branch_total = ml_df.groupby('Date')['Qty'].transform('sum')
    ml_df['branch_market_share'] = np.where(
        branch_total != 0,
        ml_df['Qty'] / branch_total,
        0.0
    )
    
    # 12. HANDLE REMAINING NaN VALUES
    print("  Handling remaining NaN values...")
    
    # Fill NaN values in lag features with 0 (no previous data)
    lag_cols = [col for col in ml_df.columns if 'lag' in col]
    ml_df[lag_cols] = ml_df[lag_cols].fillna(0)
    
    # Fill NaN values in rolling features with the current value or mean
    rolling_cols = [col for col in ml_df.columns if 'rolling' in col]
    for col in rolling_cols:
        if col.endswith('_mean') or col.endswith('_sum'):
            ml_df[col] = ml_df[col].fillna(ml_df['Qty'])
        else:
            ml_df[col] = ml_df[col].fillna(ml_df[col].mean())
    
    # Fill NaN values in expanding features
    expanding_cols = [col for col in ml_df.columns if 'expanding' in col]
    ml_df[expanding_cols] = ml_df[expanding_cols].fillna(ml_df['Qty'])
    
    # Fill NaN values in other features
    other_nan_cols = ml_df.columns[ml_df.isnull().any()].tolist()
    for col in other_nan_cols:
        if ml_df[col].dtype in ['int64', 'float64']:
            ml_df[col] = ml_df[col].fillna(ml_df[col].median())
        else:
            ml_df[col] = ml_df[col].fillna(ml_df[col].mode()[0] if not ml_df[col].mode().empty else 0)
    
    print(f"  Feature engineering completed!")
    print(f"  Total features created: {ml_df.shape[1]}")
    
    return ml_df

# Apply feature engineering
ml_df = create_ml_features(main_df)

# Display feature categories
feature_categories = {
    'Time-based': [col for col in ml_df.columns if any(x in col for x in ['year', 'month', 'day', 'week', 'quarter', 'sin', 'cos'])],
    'Lag features': [col for col in ml_df.columns if 'lag' in col],
    'Rolling statistics': [col for col in ml_df.columns if 'rolling' in col],
    'Expanding statistics': [col for col in ml_df.columns if 'expanding' in col],
    'Weather features': [col for col in ml_df.columns if any(x in col for x in ['Temp', 'Humidity', 'Wind'])],
    'Trends features': [col for col in ml_df.columns if 'trends' in col or 'Interest' in col],
    'Product features': [col for col in ml_df.columns if any(x in col for x in ['branch'])],
    'Interaction features': [col for col in ml_df.columns if 'interaction' in col],
    'Statistical features': [col for col in ml_df.columns if any(x in col for x in ['range', 'deviation', 'momentum'])],
    'Business features': [col for col in ml_df.columns if any(x in col for x in ['market_share', 'days_since'])],
}

print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  {category}: {len(features)} features")

# Check for missing values
print(f"\nMissing Values Analysis:")
missing_values = ml_df.isnull().sum()
missing_percentage = (missing_values / len(ml_df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

print(missing_summary[missing_summary['Missing Count'] > 0].head(10))


In [ ]:
# =============================================================================
# 3. DATA PREPARATION AND TRAIN-TEST SPLIT
# =============================================================================

print("\n3. DATA PREPARATION AND TRAIN-TEST SPLIT")

def prepare_ml_data(df, target_col='Qty', test_size=0.2, validation_size=0.1):
    """
    Prepare data for machine learning models with improved NaN handling
    """
    print("Preparing data for ML models...")
    
    
    # Remove rows with missing target values
    df_clean = df.dropna(subset=[target_col]).copy()
    print(f"  After removing missing target values: {len(df_clean)} samples")
    
    # Check for any remaining missing values
    missing_before = df_clean.isnull().sum().sum()
    print(f"  Missing values before cleaning: {missing_before}")
    
    # Fill any remaining missing values with appropriate strategies
    print("  Filling remaining missing values...")
    
    # Fill numerical columns with median
    numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        if col != target_col and df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            print(f"    Filled {col} with median: {df_clean[col].median():.2f}")
    
    # Fill categorical columns with mode
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().any():
            mode_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
            df_clean[col] = df_clean[col].fillna(mode_val)
            print(f"    Filled {col} with mode: {mode_val}")
    
    # Define feature columns (exclude target and non-predictive columns)
    exclude_cols = [target_col, 'Date', 'Year', 'Month', 'Week', 'Branch', 'Segment', 'Rating']
    feature_cols = [col for col in df_clean.columns if col not in exclude_cols]
    
    print(f"  Total features: {len(feature_cols)}")
    print(f"  Total samples: {len(df_clean)}")
    
    # Check final missing values
    missing_after = df_clean.isnull().sum().sum()
    print(f"  Missing values after cleaning: {missing_after}")
    
    if missing_after > 0:
        print("  Warning: Still have missing values!")
        missing_cols = df_clean.columns[df_clean.isnull().any()].tolist()
        print(f"  Columns with missing values: {missing_cols}")
    
    # Sort by date
    df_clean_sorted = df_clean.sort_values('Date')
    
    # Calculate split indices
    total_samples = len(df_clean_sorted)
    test_start_idx = int(total_samples * (1 - test_size))
    val_start_idx = int(total_samples * (1 - test_size - validation_size))
    
    # Create splits
    train_data = df_clean_sorted.iloc[:val_start_idx]
    val_data = df_clean_sorted.iloc[val_start_idx:test_start_idx]
    test_data = df_clean_sorted.iloc[test_start_idx:]
    
    # Extract features and targets for each split
    X_train = train_data[feature_cols]
    y_train = train_data[target_col]
    X_val = val_data[feature_cols]
    y_val = val_data[target_col]
    X_test = test_data[feature_cols]
    y_test = test_data[target_col]
    
    print(f"  Train set: {len(X_train)} samples ({len(X_train)/total_samples*100:.1f}%)")
    print(f"  Validation set: {len(X_val)} samples ({len(X_val)/total_samples*100:.1f}%)")
    print(f"  Test set: {len(X_test)} samples ({len(X_test)/total_samples*100:.1f}%)")
    
    # Date ranges for each split
    print(f"  Train period: {train_data['Date'].min()} to {train_data['Date'].max()}")
    print(f"  Validation period: {val_data['Date'].min()} to {val_data['Date'].max()}")
    print(f"  Test period: {test_data['Date'].min()} to {test_data['Date'].max()}")
    
    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'feature_cols': feature_cols,
        'train_data': train_data,
        'val_data': val_data,
        'test_data': test_data
    }

# Prepare the data
ml_data = prepare_ml_data(ml_df)

# Display data summary
print(f"\nData Summary:")
print(f"  Features: {len(ml_data['feature_cols'])}")
print(f"  Training samples: {len(ml_data['X_train'])}")
print(f"  Validation samples: {len(ml_data['X_val'])}")
print(f"  Test samples: {len(ml_data['X_test'])}")

# Check for any remaining missing values
print(f"\nMissing Values Check:")
print(f"  Train set missing values: {ml_data['X_train'].isnull().sum().sum()}")
print(f"  Validation set missing values: {ml_data['X_val'].isnull().sum().sum()}")
print(f"  Test set missing values: {ml_data['X_test'].isnull().sum().sum()}")

# Display feature importance preview (using correlation with target)
print(f"\nTop 10 Features by Correlation with Target:")
correlations = ml_data['X_train'].corrwith(ml_data['y_train']).abs().sort_values(ascending=False)
print(correlations.head(10))

# Save the prepared data for later use
print(f"\nSaving prepared data...")
ml_data['X_train'].to_csv('../data/ml_X_train.csv', index=False)
ml_data['X_val'].to_csv('../data/ml_X_val.csv', index=False)
ml_data['X_test'].to_csv('../data/ml_X_test.csv', index=False)
ml_data['y_train'].to_csv('../data/ml_y_train.csv', index=False)
ml_data['y_val'].to_csv('../data/ml_y_val.csv', index=False)
ml_data['y_test'].to_csv('../data/ml_y_test.csv', index=False)

print("Data preparation completed successfully!")


In [ ]:
# =============================================================================
# 4. LIGHTGBM MODEL IMPLEMENTATION
# =============================================================================

print("\n4. LIGHTGBM MODEL IMPLEMENTATION")
print("-" * 50)

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

def calculate_metrics(y_true, y_pred, model_name="Model"):
    """Calculate comprehensive evaluation metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

def train_lightgbm_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train LightGBM model with hyperparameter tuning"""
    
    print("Training LightGBM model...")
    
    # 1. Baseline LightGBM Model
    print("  Training baseline LightGBM model...")
    start_time = time.time()
    
    baseline_params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'random_state': 42
    }
    
    # Create datasets
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # Train baseline model
    baseline_model = lgb.train(
        baseline_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "LightGBM Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "LightGBM Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "LightGBM Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'num_leaves': [31, 50, 100, 200],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'feature_fraction': [0.8, 0.9, 0.95, 1.0],
        'bagging_fraction': [0.8, 0.9, 0.95, 1.0],
        'bagging_freq': [5, 10, 15],
        'min_child_samples': [20, 30, 50],
        'reg_alpha': [0, 0.1, 0.5, 1.0],
        'reg_lambda': [0, 0.1, 0.5, 1.0]
    }
    
    # Use RandomizedSearchCV for efficiency
    lgb_model = lgb.LGBMRegressor(
        objective='regression',
        metric='mae',
        boosting_type='gbdt',
        verbose=-1,
        random_state=42,
        n_estimators=1000
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        lgb_model,
        param_grid,
        n_iter=50,  # Number of parameter settings sampled
        cv=3,  # 3-fold cross-validation
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=[(X_val, y_val)],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = lgb.train(
        final_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "LightGBM Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "LightGBM Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "LightGBM Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importance(importance_type='gain')
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time
        }
    }

# Train LightGBM model
lgb_results = train_lightgbm_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nLightGBM model training completed successfully!")
print(f"Total training time: {sum(lgb_results['training_times'].values()):.2f} seconds")


In [ ]:
# =============================================================================
# 5. XGBOOST MODEL IMPLEMENTATION
# =============================================================================

print("\n5. XGBOOST MODEL IMPLEMENTATION")
print("-" * 50)

def train_xgboost_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train XGBoost model with hyperparameter tuning"""
    
    print("Training XGBoost model...")
    
    # 1. Baseline XGBoost Model
    print("  Training baseline XGBoost model...")
    start_time = time.time()
    
    baseline_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_estimators': 1000,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'verbosity': 0
    }
    
    # Train baseline model
    baseline_model = xgb.XGBRegressor(**baseline_params)
    baseline_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "XGBoost Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "XGBoost Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "XGBoost Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'max_depth': [3, 4, 5, 6, 7, 8],
        'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
        'n_estimators': [500, 800, 1000, 1200],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bylevel': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bynode': [0.6, 0.7, 0.8, 0.9, 1.0],
        'reg_alpha': [0, 0.1, 0.5, 1.0],
        'reg_lambda': [0, 0.1, 0.5, 1.0, 2.0],
        'gamma': [0, 0.1, 0.5, 1.0],
        'min_child_weight': [1, 3, 5, 7]
    }
    
    # Use RandomizedSearchCV for efficiency
    xgb_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        eval_metric='mae',
        random_state=42,
        verbosity=0
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        xgb_model,
        param_grid,
        n_iter=50,  # Number of parameter settings sampled
        cv=3,  # 3-fold cross-validation
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=[(X_val, y_val)],
                     verbose=False)
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = xgb.XGBRegressor(**final_params)
    final_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "XGBoost Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "XGBoost Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "XGBoost Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importances_
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    # 6. Advanced XGBoost Features
    print("  Training advanced XGBoost model with additional features...")
    start_time = time.time()
    
    # Advanced parameters for better performance
    advanced_params = final_params.copy()
    advanced_params.update({
        'tree_method': 'hist',  # Use histogram-based algorithm
        'grow_policy': 'lossguide',  # Grow policy for better performance
        'max_leaves': 0,  # Let max_depth control tree size
        'max_bin': 256,  # Number of bins for histogram
        'predictor': 'cpu_predictor',  # Use CPU predictor
        'enable_categorical': False,  # Disable categorical features
        'interaction_constraints': None,  # No interaction constraints
        'monotone_constraints': None,  # No monotone constraints
    })
    
    # Train advanced model
    advanced_model = xgb.XGBRegressor(**advanced_params)
    advanced_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    advanced_time = time.time() - start_time
    
    # Make predictions with advanced model
    advanced_train_pred = advanced_model.predict(X_train)
    advanced_val_pred = advanced_model.predict(X_val)
    advanced_test_pred = advanced_model.predict(X_test)
    
    # Calculate advanced metrics
    advanced_train_metrics = calculate_metrics(y_train, advanced_train_pred, "XGBoost Advanced Train")
    advanced_val_metrics = calculate_metrics(y_val, advanced_val_pred, "XGBoost Advanced Val")
    advanced_test_metrics = calculate_metrics(y_test, advanced_test_pred, "XGBoost Advanced Test")
    
    print(f"    Advanced training time: {advanced_time:.2f} seconds")
    print(f"    Advanced validation RMSE: {advanced_val_metrics['RMSE']:.4f}")
    print(f"    Advanced validation R²: {advanced_val_metrics['R2']:.4f}")
    
    # Compare all XGBoost models
    print("  All XGBoost models comparison:")
    print(f"    Baseline RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Final RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Advanced RMSE: {advanced_val_metrics['RMSE']:.4f}")
    
    best_model = 'Advanced' if advanced_val_metrics['RMSE'] < final_val_metrics['RMSE'] else 'Final'
    print(f"    Best XGBoost model: {best_model}")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'advanced_model': advanced_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'advanced_metrics': {
            'train': advanced_train_metrics,
            'val': advanced_val_metrics,
            'test': advanced_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            },
            'advanced': {
                'train': advanced_train_pred,
                'val': advanced_val_pred,
                'test': advanced_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time,
            'advanced': advanced_time
        }
    }

# Train XGBoost model
xgb_results = train_xgboost_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nXGBoost model training completed successfully!")
print(f"Total training time: {sum(xgb_results['training_times'].values()):.2f} seconds")


In [ ]:
# =============================================================================
# 6. CATBOOST MODEL IMPLEMENTATION
# =============================================================================

print("\n6. CATBOOST MODEL IMPLEMENTATION")
print("-" * 50)

def train_catboost_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train CatBoost model with hyperparameter tuning"""
    
    print("Training CatBoost model...")
    
    # 1. Baseline CatBoost Model
    print("  Training baseline CatBoost model...")
    start_time = time.time()
    
    baseline_params = {
        'iterations': 1000,
        'learning_rate': 0.1,
        'depth': 6,
        'l2_leaf_reg': 3,
        'bootstrap_type': 'Bayesian',
        'random_seed': 42,
        'od_type': 'Iter',
        'od_wait': 100,
        'verbose': False
    }
    
    # Train baseline model
    baseline_model = cb.CatBoostRegressor(**baseline_params)
    baseline_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "CatBoost Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "CatBoost Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "CatBoost Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'iterations': [500, 800, 1000, 1200],
        'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
        'depth': [4, 5, 6, 7, 8],
        'l2_leaf_reg': [1, 3, 5, 7, 9],
        'bootstrap_type': ['Bayesian', 'Bernoulli'],
        'bagging_temperature': [0, 0.5, 1.0],
        'random_strength': [0, 1, 2],
        'one_hot_max_size': [2, 10, 20],
        'leaf_estimation_method': ['Newton', 'Gradient'],
        'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide']
    }
    
    # Use RandomizedSearchCV for efficiency
    cb_model = cb.CatBoostRegressor(
        random_seed=42,
        od_type='Iter',
        od_wait=100,
        verbose=False
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        cb_model,
        param_grid,
        n_iter=30,  # Number of parameter settings sampled
        cv=3,  # 3-fold cross-validation
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=(X_val, y_val),
                     early_stopping_rounds=100,
                     verbose=False)
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = cb.CatBoostRegressor(**final_params)
    final_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "CatBoost Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "CatBoost Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "CatBoost Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importances_
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time
        }
    }

# Train CatBoost model
cb_results = train_catboost_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nCatBoost model training completed successfully!")
print(f"Total training time: {sum(cb_results['training_times'].values()):.2f} seconds")
